# 7-2절 연습 문제 풀이

이 노트북은 7-2절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch07/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 7장 공통 - 텍스트 로더와 OzWriter 모델
from torch.utils.data import DataLoader, TensorDataset, random_split
import re

def load_text(path, strip_gutenberg=True):
    with open(path, encoding='utf-8-sig') as f:
        text = f.read()
    if strip_gutenberg:
        s = text.find('*** START')
        e = text.find('*** END')
        if s != -1: text = text[text.find('\n', s) + 1:]
        if e != -1: text = text[:text.find('*** END')]
    return text

def tokenize(text):
    text = text.lower().replace('\n', ' ')
    text = re.sub(r'([.,!?])', r' \1 ', text)
    return [t for t in text.split() if t]

def make_seq_dataset(tokens, vocab, window=8):
    idx = torch.tensor([vocab[t] for t in tokens])
    xs = torch.stack([idx[i:i + window] for i in range(len(idx) - window)])
    ys = idx[window:]
    return TensorDataset(xs, ys)

class OzWriter(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)
    def forward(self, x):
        out, _ = self.rnn(self.emb(x))
        return self.fc(out[:, -1, :])

## 연습 7-4

OzWriter 모델의 학습 루프와 문장 생성 함수를 직접 만들어 보자. 참고로 7-2절의 깃허브 예제 노트북을 조금만 내리면 저자가 작성한 예제 코드가 있지만, 이를 확인하기 전 5장과 6장에서 학습한 다음의 기능을 구현할 수 있는지 직접 테스트해 보자.

지정한 CPU 또는 하드웨어 가속기 장치를 사용하는 학습과 예측

순환 신경망 모델의 학습 루프

정해진 수의 토큰을 생성해 문장을 만드는 생성 함수. 단, 비정상 종료되지 않도록 예외 처리가 되어 있어야 한다.

In [ ]:
tokens = tokenize(load_text('../../data/wonderful_wizard_of_oz.txt'))[:20000]
vocab = {t: i for i, t in enumerate(sorted(set(tokens)))}
rev = {i: t for t, i in vocab.items()}
loader = DataLoader(make_seq_dataset(tokens, vocab), batch_size=64, shuffle=True)
print(f'토큰 {len(tokens):,}개, 어휘 {len(vocab):,}개')

def train_writer(model, loader, epochs=5, lr=2e-3):
    model = model.to(device)                      # 1) 지정한 장치 사용
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    log = common.EpochLogger(epochs, columns=('훈련 손실',), formats=('{:.4f}',))
    for epoch in range(1, epochs + 1):            # 2) 에포크 반복
        model.train(); tot = n = 0
        for x, y in loader:                       # 3) 배치 학습
            x, y = x.to(device), y.to(device)
            loss = criterion(model(x), y)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tot += loss.item() * len(y); n += len(y)
        log.row(epoch, tot / n)                   # 4) 학습 로그
    log.summary()
    return model

torch.manual_seed(SEED)
model = train_writer(OzWriter(len(vocab)), loader)

In [ ]:
@torch.no_grad()
def generate(model, seed_text, n=30, window=8, temperature=0.8):
    model.eval()
    out = tokenize(seed_text)
    for _ in range(n):
        ids = [vocab.get(t, 0) for t in out[-window:]]
        ids = [0] * (window - len(ids)) + ids
        logits = model(torch.tensor([ids], device=device))
        probs = torch.softmax(logits[0] / temperature, dim=-1)
        out.append(rev[torch.multinomial(probs, 1).item()])
    return ' '.join(out)

print(generate(model, 'dorothy lived in the'))

학습 루프에 필요한 요소는 ① 장치 이동(`to(device)`), ② 에포크 반복, ③ 배치 단위 순전파·역전파, ④ 로그 출력이다. 생성 함수는 마지막 `window`개 토큰으로 다음 토큰을 예측하고, 그 결과를 다시 입력에 이어 붙이는 자기회귀 방식이다.

## 연습 7-5

6장 [연습 문제 6-12]와 마찬가지로 소설 <오즈의 마법사>의 첫 부분과 끝 부분에 위치한 소설과 무관한 내용을 삭제한 후 모델을 다시 학습하고 생성 결과를 확인해 보자.

In [ ]:
raw = load_text('../../data/wonderful_wizard_of_oz.txt', strip_gutenberg=False)
clean = load_text('../../data/wonderful_wizard_of_oz.txt', strip_gutenberg=True)
print(f'원본 {len(raw):,}자 -> 정리 후 {len(clean):,}자 (머리말·꼬리말 {len(raw) - len(clean):,}자 제거)')

tokens_c = tokenize(clean)[:20000]
vocab_c = {t: i for i, t in enumerate(sorted(set(tokens_c)))}
rev_c = {i: t for t, i in vocab_c.items()}
loader_c = DataLoader(make_seq_dataset(tokens_c, vocab_c), batch_size=64, shuffle=True)
torch.manual_seed(SEED)
model_c = train_writer(OzWriter(len(vocab_c)), loader_c)
vocab, rev = vocab_c, rev_c
print('\n' + generate(model_c, 'dorothy lived in the'))

라이선스 안내문에는 소설에 없는 법률 용어(`project`, `gutenberg`, `license` 등)가 들어 있어 어휘 사전을 오염시키고, 생성 결과에도 섞여 나온다. 제거하면 어휘가 줄고 문체가 일관돼 생성 품질이 올라간다.

## 연습 7-6

소설 <오즈의 마법사>의 데이터셋을 만들 때, 전체의 10% 분량을 검증 데이터셋으로 분리해 모델을 학습시켜 보자. 에포크별 훈련 손실과 검증 손실을 확인하고 다음 질문에 답해 보자.

과적합 여부를 확인할 수 있는가?

확인 결과를 바탕으로 검증 손실과 평가 데이터셋으로 계산한 지표로 생성 모델의 성능 확인이 힘든 이유를 설명해 보자.

In [ ]:
dataset = make_seq_dataset(tokens_c, vocab_c)
n_val = int(len(dataset) * 0.1)
tr, va = random_split(dataset, [len(dataset) - n_val, n_val],
                      generator=torch.Generator().manual_seed(SEED))
tr_loader = DataLoader(tr, batch_size=64, shuffle=True)
va_loader = DataLoader(va, batch_size=64)

torch.manual_seed(SEED)
model_v = OzWriter(len(vocab_c)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_v.parameters(), lr=2e-3)
for epoch in range(1, 11):
    model_v.train(); t = n = 0
    for x, y in tr_loader:
        x, y = x.to(device), y.to(device)
        loss = criterion(model_v(x), y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        t += loss.item() * len(y); n += len(y)
    model_v.eval(); vt = vn = 0
    with torch.no_grad():
        for x, y in va_loader:
            x, y = x.to(device), y.to(device)
            vt += criterion(model_v(x), y).item() * len(y); vn += len(y)
    print(f'{epoch:2d} 훈련 {t / n:.4f} / 검증 {vt / vn:.4f}')

**과적합 확인 가능 여부**: 가능하다. 훈련 손실은 계속 줄지만 검증 손실은 어느 시점부터 올라가는 전형적인 패턴이 나타난다.

**생성 모델의 성능 확인이 어려운 이유**: 검증 손실이 낮다는 것은 '다음 단어를 잘 맞힌다'는 뜻일 뿐, **생성된 문장이 자연스러운지와는 다르다**. 창작에는 정답이 하나가 아니어서, 원문과 다른 단어를 골라도 훌륭한 문장일 수 있는데 손실은 이를 벌점으로 계산한다.

그래서 생성 모델은 손실 외에 사람이 직접 읽어 보는 정성 평가나, 13장에서 언급한 BLEU·METEOR 같은 별도 지표를 함께 사용한다.

## 연습 7-7

임베딩 계층을 사용하는 OzWriter 모델은 OzRawWriter 모델에 비해 학습 속도가 빠르다. 이는 일반적인 현상인데, 그 이유를 데이터 처리 방식과 모델이 데이터를 이해하는 방식으로 나누어 제시해 보자.

7-8 소설 <오즈의 마법사>가 미국을 대표하는 환상 소설 중 하나라면, 루이스 캐럴Lewis Carroll이 1865년 발표한 <이상한 나라의 앨리스Alice's Adventures in Wonderland>는 영국을 대표하는 환상 소설 중 하나이다. 마찬가지로 구텐베르크 프로젝트에서 저작권 없는 텍스트 파일을 내려받을 수 있다(https://www.gutenberg.org/cache/epub/11/pg11.txt). 이 파일을 내려받아 전처리한 후 소설 <오즈의 마법사>의 데이터와 합쳐 OzWriter 모델을 학습한 후, 두 소설에 나오는 문자열을 포함해 다양한 마중물 텍스트의 생성 결과를 확인해 보자.

7-9 [도전 문제] [연습 문제 7-8]에서 학습한 모델에 포함된 임베딩 계층을 사용해 어휘 사전에 포함된 단어 사이의 코사인 유사도 상위 50개 쌍을 출력해 보자. 이때 어휘 사전에서 다음과 같은 상용 단어는 비교 어휘에서 제외한다.

*코드 7-6 상용 단어 리스트*

```python
stopwords = {
    'the', 'a', 'an', 'and', 'or', 'of', 'to', 'in', 'on', 'at',
    'is', 'was', 'are', 'were', 'be', 'been', 'being',
    'he', 'she', 'it', 'they', 'we', 'i', 'you', 'me', 'him', 'her',
    'them', 'us', 'his', 'their', 'its', 'my', 'your', 'our',
    'this', 'that', 'these', 'those',
    'as', 'with', 'for', 'by', 'from', 'but', 'so', 'if', 'than',
    'not', 'no', 'do', 'did', 'does', 'have', 'has', 'had',
    'will', 'would', 'shall', 'should', 'can', 'could', 'may', 'might',
}
```

코사인 유사도는 다음과 같이 두 가지 방법으로 각각 계산해 보자.

두 텐서의 코사인 유사도를 계산하는 torch.nn.functional.cosine_similarity()를 사용해 구현한다.

torch.nn.functional.cosine_similarity()를 사용하지 않고 직접 구현한다. 코사인 유사도(cosine similarity)는 두 텐서의 방향이 얼마나 비슷한지를 나타내는 척도로, 텐서 요솟값의 크기는 무시하고 원점에서 두 텐서가 가리키는 점까지 선을 그었을 때 두 선 사이의 각도의 코사인 값이 코사인 유사도가 된다. 예를 들어 torch.tensor([0., 1.])과 torch.tensor([100., 0.])은 평면 좌표에서 직각(90°)을 이루는 두 축 위에 각각 자리 잡으므로, 두 텐서의 코사인 유사도는 cos(90°)=0이다.

### 풀이

**데이터 처리 방식**
원-핫 입력은 (배치, 윈도우, 어휘 수) 크기의 거대한 희소 행렬을 만들어 곱셈을 수행한다. 임베딩은 인덱스로 **행 하나를 조회**할 뿐이라 곱셈 자체가 없다. 어휘가 1만 개면 1만 번의 곱셈이 조회 한 번으로 바뀐다. 메모리 사용량도 크게 줄어 배치를 키울 수 있다.

**모델이 데이터를 이해하는 방식**
원-핫에서는 모든 단어가 서로 직교해 아무 관계도 없는 상태에서 출발한다. 모델은 단어 사이의 관계를 처음부터 전부 배워야 한다. 임베딩은 학습이 진행되며 **비슷한 단어가 비슷한 벡터**가 되므로, 한 단어에서 배운 것이 유사한 단어에도 함께 적용된다. 같은 데이터로 더 많이 배우는 셈이라 수렴이 빠르다.

## 연습 7-8

<이상한 나라의 앨리스> 텍스트(https://www.gutenberg.org/cache/epub/11/pg11.txt)를 내려받아 전처리한 후 <오즈의 마법사> 데이터와 합쳐 OzWriter 모델을 학습하고, 다양한 마중물 텍스트의 생성 결과를 확인해 보자.

In [ ]:
# 두 소설을 합쳐 학습한다. (파일이 없으면 안내만 출력)
import os, urllib.request
ALICE_PATH = '../../data/alice_in_wonderland.txt'
if not os.path.exists(ALICE_PATH):
    print('알림: alice_in_wonderland.txt 가 없어 내려받는다.')
    urllib.request.urlretrieve('https://www.gutenberg.org/cache/epub/11/pg11.txt',
                               ALICE_PATH)

alice = tokenize(load_text(ALICE_PATH))
oz = tokenize(load_text('../../data/wonderful_wizard_of_oz.txt'))
both = (oz + alice)[:40000]
vocab_b = {t: i for i, t in enumerate(sorted(set(both)))}
rev_b = {i: t for t, i in vocab_b.items()}
print(f'오즈 {len(oz):,} + 앨리스 {len(alice):,} -> 사용 {len(both):,} 토큰, '
      f'어휘 {len(vocab_b):,}개')

loader_b = DataLoader(make_seq_dataset(both, vocab_b), batch_size=64, shuffle=True)
torch.manual_seed(SEED)
model_b = train_writer(OzWriter(len(vocab_b)), loader_b, epochs=8)
vocab, rev = vocab_b, rev_b
for seed in ['dorothy lived in the', 'alice was beginning to', 'the scarecrow said']:
    print(f'\n{seed!r}\n  -> {generate(model_b, seed, n=25)}')

두 소설을 합치면 어휘와 문체가 다양해져 특정 작품을 그대로 외우기 어려워진다. 대신 두 작품의 등장인물이 한 문장에 섞여 나오는 결과도 나타난다. 모델에 어느 작품인지 알려 주는 단서가 없기 때문인데, 이런 조건을 주는 방법이 12장의 프롬프트다.

## 연습 7-9

[도전 문제] [연습 문제 7-8]에서 학습한 모델의 임베딩 계층으로 어휘 사전 단어 사이의 코사인 유사도 상위 50개 쌍을 출력해 보자. 상용 단어([코드 7-6]의 stopwords)는 제외한다.

In [ ]:
stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'if', 'of', 'to', 'in', 'on',
             'at', 'by', 'for', 'with', 'is', 'was', 'were', 'be', 'been', 'it',
             'he', 'she', 'they', 'you', 'i', 'his', 'her', 'their', 'this',
             'that', 'said', 'not', 'as', 'so', 'then', 'there', 'had', 'have',
             '.', ',', '!', '?'}

emb = model_b.emb.weight.detach()                     # (어휘 수, 임베딩 차원)
keep = [i for t, i in vocab_b.items() if t not in stopwords and len(t) > 2]
sub = nn.functional.normalize(emb[keep], dim=1)       # L2 정규화 -> 내적 = 코사인 유사도
sim = sub @ sub.T
sim.fill_diagonal_(-1)                                # 자기 자신 제외

pairs = torch.triu(sim, diagonal=1)                   # 위쪽 삼각행렬만(중복 쌍 제거)
values, flat = pairs.flatten().topk(50)
for rank, (v, f) in enumerate(zip(values.tolist(), flat.tolist()), start=1):
    i, j = divmod(f, len(keep))
    print(f'{rank:2d}. {rev_b[keep[i]]:15s} - {rev_b[keep[j]]:15s} {v:.4f}')

임베딩 벡터를 **L2 정규화**하면 내적이 곧 코사인 유사도가 된다. 위쪽 삼각행렬만 남겨 (A,B)와 (B,A) 중복을 없앴다.

학습 데이터가 소설 두 편뿐이라 유사도 상위 쌍은 의미가 비슷한 단어보다 **같은 문맥에 자주 붙어 나오는 단어**인 경우가 많다. 대규모 말뭉치로 학습한 word2vec이나 GloVe는 훨씬 뚜렷한 의미 관계를 보인다.